# SFT → DPO → Eval on Colab Pro A100

Run this AFTER `colab_pretrain_60m.ipynb` produced `checkpoints/base_60m/final.pt`.

End-to-end pipeline:
1. Build SFT dataset from TinyStoriesInstruct
2. Train SFT (~1 hour A100)
3. Sample candidate preference pairs from SFT model (~1 hour)
4. Label preferences with Qwen-2.5-3B-Instruct judge (~1 hour)
5. Train DPO (~2 hours)
6. Full eval matrix → results.md (~40 min)

**Total time on A100:** ~6 hours. Set this overnight.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/transformer-lm'
for d in ['checkpoints/sft', 'checkpoints/dpo', 'data', 'logs']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)

In [ ]:
%cd /content
!git clone https://github.com/Qwerty0826/char-level-transformer-lm.git || (cd char-level-transformer-lm && git pull)
%cd /content/char-level-transformer-lm
!pip install -e . -q
!pip install datasets transformers bitsandbytes accelerate -q

!rm -rf data checkpoints logs
!ln -sf {DRIVE_ROOT}/data        data
!ln -sf {DRIVE_ROOT}/checkpoints checkpoints
!ln -sf {DRIVE_ROOT}/logs        logs
!ls -la checkpoints/

In [ ]:
# Verify the base pretrain finished and produced a final.pt.
assert os.path.exists('checkpoints/base_60m/final.pt'), \
    'Base pretrain checkpoint missing. Run colab_pretrain_60m.ipynb first.'
print('Base checkpoint:', os.path.getsize('checkpoints/base_60m/final.pt') / 1e6, 'MB')

## 1. Build the SFT dataset (~5 min)

In [ ]:
!python scripts/build_sft_dataset.py \
    --vocab  data/tinystories_v2_vocab.json \
    --merges data/tinystories_v2_merges.txt \
    --output data/tinystories_v2_sft.pt \
    --max_length 512 --max_examples 40000

## 2. SFT training (~1 hour A100)

In [ ]:
!python scripts/train_sft.py \
    --base_checkpoint checkpoints/base_60m/final.pt \
    --sft_data        data/tinystories_v2_sft.pt \
    --checkpoint_dir  checkpoints/sft \
    --total_steps 2000 --batch_size 32 \
    --lr_max 3e-5 --warmup_steps 100 \
    --device cuda --dtype bfloat16 --compile

In [ ]:
# Manual sanity check: does the SFT model respect the chat template?
import torch
from cs336_basics.model import TransformerLM
from cs336_basics.tokenizer import Tokenizer
from cs336_basics.data_sft import USER_TAG, ASSISTANT_TAG, EOT

tok = Tokenizer.from_files(
    'data/tinystories_v2_vocab.json',
    'data/tinystories_v2_merges.txt',
    special_tokens=['<|endoftext|>', '<|user|>', '<|assistant|>', '<|system|>'],
)
model = TransformerLM(
    vocab_size=16000, context_length=512, d_model=640, num_layers=10,
    num_heads=10, num_kv_heads=2, d_ff=1728, device='cuda', dtype=torch.bfloat16,
)
ckpt = torch.load('checkpoints/sft/final.pt', map_location='cpu')
state = ckpt.get('model_state_dict', ckpt)
if any(k.startswith('_orig_mod.') for k in state):
    state = {k.removeprefix('_orig_mod.'): v for k, v in state.items()}
model.load_state_dict(state, strict=False)
model.eval()

eot = tok.encode(EOT)[0]
prompt = USER_TAG + 'Write a story about a fox who learns to share.' + EOT + ASSISTANT_TAG
ids = torch.tensor([tok.encode(prompt)], device='cuda')
out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_p=0.95, eos_id=eot)
print(tok.decode(out[0].tolist()))

## 3. Build candidate preference pairs (~1 hour)

Samples 2 completions per held-out prompt at T=0.9, top-p=0.95. Heuristic-filters pairs that are too similar to provide DPO signal.

In [ ]:
!python scripts/build_preference_dataset.py \
    --sft_checkpoint checkpoints/sft/final.pt \
    --vocab          data/tinystories_v2_vocab.json \
    --merges         data/tinystories_v2_merges.txt \
    --output         data/pref_candidates.jsonl \
    --num_prompts 1000 --max_new_tokens 200 \
    --device cuda --dtype bfloat16

## 4. Label preferences with local Qwen judge (~1 hour)

Each pair is judged in BOTH orders. Pairs where the judge flips are dropped (position-bias control). Aim for ≥300 high-confidence pairs.

In [ ]:
!python scripts/label_preferences.py \
    --input  data/pref_candidates.jsonl \
    --output data/pref_labeled.jsonl \
    --judge_model Qwen/Qwen2.5-3B-Instruct \
    --device cuda --load_in_4bit

## 5. DPO training (~2 hours A100)

Loads the SFT checkpoint twice (policy trainable, ref frozen). β=0.1. Watch `margin` — it must trend up over training.

In [ ]:
!python scripts/train_dpo.py \
    --sft_checkpoint checkpoints/sft/final.pt \
    --preferences    data/pref_labeled.jsonl \
    --vocab          data/tinystories_v2_vocab.json \
    --merges         data/tinystories_v2_merges.txt \
    --checkpoint_dir checkpoints/dpo \
    --total_steps 1500 --batch_size 4 \
    --lr_max 5e-6 --beta 0.1 \
    --device cuda --dtype bfloat16

## 6. Full eval matrix → results.md (~40 min)

In [ ]:
!python scripts/eval_all.py \
    --base_checkpoint checkpoints/base_60m/final.pt \
    --sft_checkpoint  checkpoints/sft/final.pt \
    --dpo_checkpoint  checkpoints/dpo/final.pt \
    --val_data        data/tinystories_v2_tokens_val.npy \
    --vocab           data/tinystories_v2_vocab.json \
    --merges          data/tinystories_v2_merges.txt \
    --output          results.md \
    --num_eval_prompts 150 \
    --judge_model Qwen/Qwen2.5-3B-Instruct \
    --device cuda --load_in_4bit

In [ ]:
# Read the results back so you can see them in this session.
with open('results.md') as f:
    print(f.read())

# Copy results.md back to Drive for safekeeping.
!cp results.md {DRIVE_ROOT}/results.md
print('Saved to Drive:', f'{DRIVE_ROOT}/results.md')

## Done.

Headlines to inspect in `results.md`:
- **DPO beats SFT in X% of pairs.** If X ≥ 58% on 150 prompts with swap-consistency ≥ 70%, you have a defensible resume bullet.
- If DPO win-rate is in the 50-55% noise band, the implementation is still demonstrably correct (test suite passes, reward margin trended up during training) — the eval just couldn't detect a small effect at this scale. Report the result honestly in the README.

Next: commit `results.md` to the repo and update the README with the headline numbers.